In [24]:
## mount google colab
from google.colab import drive
drive.mount('/content/gdrive')
%cd /content/gdrive/My Drive/Digital Transformation Notebooks/Data

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
/content/gdrive/My Drive/Digital Transformation Notebooks/Data


In [3]:
!pip install -q pyomo
!apt-get install -y -qq glpk-utils

Selecting previously unselected package libsuitesparseconfig5:amd64.
(Reading database ... 123633 files and directories currently installed.)
Preparing to unpack .../libsuitesparseconfig5_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libsuitesparseconfig5:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libamd2:amd64.
Preparing to unpack .../libamd2_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libamd2:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libcolamd2:amd64.
Preparing to unpack .../libcolamd2_1%3a5.10.1+dfsg-4build1_amd64.deb ...
Unpacking libcolamd2:amd64 (1:5.10.1+dfsg-4build1) ...
Selecting previously unselected package libglpk40:amd64.
Preparing to unpack .../libglpk40_5.0-1_amd64.deb ...
Unpacking libglpk40:amd64 (5.0-1) ...
Selecting previously unselected package glpk-utils.
Preparing to unpack .../glpk-utils_5.0-1_amd64.deb ...
Unpacking glpk-utils (5.0-1) ...
Setting up libsuitesparseconfig5:amd64 (1:5.10.1+dfsg-4b


# Shipping Optimization with Pyomo

Imagine that you are brought in as a consultant for a commodity distribution company. The company is hoping to minimize the transportation costs of shipping their product from two factories, located in Austin and Los Angeles to customers in 6 other cities, by choosing how many units from each factory to send to each city. The company must fulfill the demand at each of these 6 cities, while not exceeding the supply available at their two factory locations. The total cost is the sum over all shipments between factories and cities of the unit transportation cost times the number of units shipped. Using the demand, supply, and unit transportation costs for each city found in
`transport_network_data.csv`, you want to set up and solve an integer linear optimization problem to decide a distribution strategy.

Let's first look at the data:

In [25]:
import pandas as pd
network_data = pd.read_csv('transport_network_data.csv')
network_data

,Customer\Source,Austin [$/ton],Los Angeles [$/ton],Demand [tons]
0,Seattle,NaN,2.1,110.0
1,Kansas City,2.3,5.5,170.0
2,Salt Lake City,1.6,2.2,211.0
3,Phoenix,1.3,0.9,245.0
4,Las Vegas,1.2,1.2,230.0
5,San Francisco,0.5,0.7,210.0
6,Supply [tons],600.0,750.0,NaN


This data file gives us the costs of shipping to each customer city (rows) from the factories in Austin and Los Angeles (columns),  as well as the demand in each city and the supply available in each factory. Cost of shipping from Austin to Seattle is not given ("NaN") - we will assume this is impossible, and so will set a very high price for this shipment: let's choose $1000 per ton. Our cost minimizing model will thus avoid this possibility. We can do this using the `.iloc[rows,columns]` syntax. We want to change row 0, column 1 to 1000 (remember that Pandas dataframes indices start at 0), which we do as follows:

In [26]:
network_data.iloc[0,1]=1000
network_data



,Customer\Source,Austin [$/ton],Los Angeles [$/ton],Demand [tons]
0,Seattle,1000.0,2.1,110.0
1,Kansas City,2.3,5.5,170.0
2,Salt Lake City,1.6,2.2,211.0
3,Phoenix,1.3,0.9,245.0
4,Las Vegas,1.2,1.2,230.0
5,San Francisco,0.5,0.7,210.0
6,Supply [tons],600.0,750.0,NaN


Next, lets extract just the costs in a variable - rows 0 to 5 (remember that Python does not include the last index given in a slice, so this is written `0:6`), and columns 1 to 2:

In [27]:
import numpy as np
Costs = np.array(network_data.iloc[0:6,1:3])
print(Costs)

[[1.0e+03 2.1e+00]
 [2.3e+00 5.5e+00]
 [1.6e+00 2.2e+00]
 [1.3e+00 9.0e-01]
 [1.2e+00 1.2e+00]
 [5.0e-01 7.0e-01]]


Similarly, we extract the Demands for each city:

In [28]:
Demands = np.array(network_data.iloc[0:6,3])
print(Demands)

[110. 170. 211. 245. 230. 210.]


And the Supplies of each factory:

In [29]:
Supply = np.array(network_data.iloc[6,1:3])
print(Supply)

[600.0 750.0]



Let's now formulate the problem mathematically. We have a set of factories $f \in F$, and a set of cities/customers $c \in C$ to supply. Our decision variables $x_{cf}$ will be the quantity shipped from factory $f$ to city $$. These will all be non-negative integer variables.

We want to minimize our total shipping costs, this be the sum all shipping costs per unit for each city-factory combination times the number of units shipped  for that combination. In math, we have:

$$\sum_{c \in \text{Cities}} \sum_{f \in \text{Factories}} Costs_{cf} x_{cf}$$

We also need to meet all demand for each city, summing up over the amount provided by each factory. We can write this follows:

$$\sum_{f \in \text{Factories}} x_{cf} = \text{Demands}_c \quad \forall c \in \text{Cities}$$

Finally, we need to make sure the total sent to all cities from any given factory is not greater than that factory's supply:

$$
\sum_{c \in \text{Cities}} x_{cf} \leq \text{Supply}_f \quad \forall f \in \text{Factories}
$$

To put this in Pyomo, we first define our indices (Cities and Factories), create our decision variable (for each city and factory, as a non-negative integer), and finally, write our objective function, which is a minimization (so we set `sense=Minimize`). Run the next cell to define these elements.


In [30]:
from pyomo.environ import *

# create a model
model = ConcreteModel()

#supply/factory indices
F = range(2)
#demand/city indices
C= range(6)


# declare decision variables
model.x = Var(C,F,domain=NonNegativeIntegers )

# declare objective
model.min_cost = Objective(expr =  sum(Costs[c,f]*model.x[c,f] for c in C for f in F), sense=minimize)


Next, we have our constraints. First, we will initialize a constraint list, then put define each of our constraints as elements of this list:

In [31]:
# add model constraints
model.constraints = ConstraintList()

# supply each demand node adequately (sum of supplys from each warehouse must equal total demand for customer)
for c in C:
  model.constraints.add(sum(model.x[c,f] for f in F) ==Demands[c])

# do not deliver more supply than available (sum of supplys for each customer must be less than supply for warehouse)
for f in F:
  model.constraints.add(sum(model.x[c,f] for c in C) <=Supply[f])

Now we have all the elements we need to solve the problem:

In [32]:
SolverFactory('glpk', executable='/usr/bin/glpsol').solve(model).write()

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Name: unknown
  Lower bound: 1561.1
  Upper bound: 1561.1
  Number of objectives: 1
  Number of constraints: 8
  Number of variables: 12
  Number of nonzeros: 24
  Sense: minimize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 1
      Number of created subproblems: 1
  Error rc: 0
  Time: 0.00497126579284668
# ----------------------------------------------------------
#   Solution Information
# ---------------------------------

Let's print out our model cost and the amount to ship from each factory to each city. This gives our solution to the optimization problem.

In [34]:
# display solution
print('\nCost = ', model.min_cost())

print('\nDecision Variables')


Cities = network_data.iloc[0:6,0]
Factories = ['Austin', 'Los Angeles']

#print x
print('\nx: flow from factories to cities:')
for f in F:
  for c in C:
    amount = model.x[c,f].value
    print("From Factory %s to City %s:" % (Factories[f], Cities[c]), amount )





Cost =  1561.1

Decision Variables

x: flow from factories to cities:
From Factory Austin to City Seattle: 0.0
From Factory Austin to City Kansas City: 170.0
From Factory Austin to City Salt Lake City: 211.0
From Factory Austin to City Phoenix: 0.0
From Factory Austin to City Las Vegas: 0.0
From Factory Austin to City San Francisco: 210.0
From Factory Los Angeles to City Seattle: 110.0
From Factory Los Angeles to City Kansas City: 0.0
From Factory Los Angeles to City Salt Lake City: 0.0
From Factory Los Angeles to City Phoenix: 245.0
From Factory Los Angeles to City Las Vegas: 230.0
From Factory Los Angeles to City San Francisco: 0.0


Looking at the solution values for shipments from the Los Angeles factory, it looks like the total shipped (110+245+230=585) is substantially less than our supply at that factory (750). This suggests that we might be able to save money by reducing our supply at the Los Angeles factory.

Let's try setting supply at Los Angeles at 600 (leaving some buffer for emergencies) and make sure that doesn't increase our costs:


In [38]:
Supply[1] = 600

Now, resolve the model:

In [44]:
from pyomo.environ import *

# create a model
model = ConcreteModel()

#supply/factory indices
F = range(2)
#demand/city indices
C= range(6)


# declare decision variables
model.x = Var(C,F,domain=NonNegativeIntegers )

# declare objective
model.min_cost = Objective(expr =  sum(Costs[c,f]*model.x[c,f] for c in C for f in F), sense=minimize)

# add model constraints
model.constraints = ConstraintList()

# supply each demand node adequately (sum of supplys from each warehouse must equal total demand for customer)
for c in C:
  model.constraints.add(sum(model.x[c,f] for f in F) ==Demands[c])

# do not deliver more supply than available (sum of supplys for each customer must be less than supply for warehouse)
for f in F:
  model.constraints.add(sum(model.x[c,f] for c in C) <=Supply[f])

SolverFactory('glpk', executable='/usr/bin/glpsol').solve(model).write()

# display solution
print('\nCost = ', model.min_cost())

print('\nDecision Variables')


Cities = network_data.iloc[0:6,0]
Factories = ['Austin', 'Los Angeles']

#print x
for f in F:
  for c in C:
    amount = model.x[c,f].value
    print("From Factory %s to City %s:" % (Factories[f], Cities[c]), amount )


# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Name: unknown
  Lower bound: 1561.1
  Upper bound: 1561.1
  Number of objectives: 1
  Number of constraints: 8
  Number of variables: 12
  Number of nonzeros: 24
  Sense: minimize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 1
      Number of created subproblems: 1
  Error rc: 0
  Time: 0.004780769348144531
# ----------------------------------------------------------
#   Solution Information
# --------------------------------

As expected, the solution remains exactly the same. What happens if we switch the Los Angeles supply to 580? What about 550? Try it out!